In [6]:
# Code Block 0 — Setup, Auth, BigQuery location autodetect, and helpers
!pip -q install plotly pandas-gbq google-cloud-bigquery pyarrow gcsfs

from google.colab import auth
auth.authenticate_user()

import pandas as pd
import numpy as np
import pandas_gbq
from google.cloud import bigquery
import plotly.io as pio
pio.renderers.default = "colab"

import textwrap, warnings
warnings.filterwarnings("ignore")

# ---- Project config ----
PROJECT_ID  = "mgmt590-465220"
DATASET_ID  = "final_clean"
BUCKET_NAME = "sameeryelamarthi-final"  # for future GCS exports

# ---- BigQuery client & dataset location detection ----
client = bigquery.Client(project=PROJECT_ID)

try:
    ds = client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
    BQ_LOCATION = ds.location or "US"  # e.g., "US", "EU", or regional like "us-central1"
    print(f"✅ Dataset found: {ds.full_dataset_id} | location = {BQ_LOCATION}")

    tables = list(client.list_tables(ds))
    TABLE_IDS = [t.table_id for t in tables]
    print(f"Found {len(TABLE_IDS)} tables (first 25 shown): {TABLE_IDS[:25]}")
except Exception as e:
    print("❌ Couldn't find dataset. Details:\n", e)
    print("Datasets visible in this project:")
    try:
        print([d.dataset_id for d in client.list_datasets(project=PROJECT_ID)])
    except Exception as ee:
        print("Also couldn't list datasets:", ee)
    raise

# ---- Helpers ----
def t(name: str) -> str:
    """Fully-qualified table reference for the configured dataset."""
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

def run_gbq(sql: str, *, location: str = None) -> pd.DataFrame:
    """Read a SQL query from BigQuery, forcing the correct location."""
    loc = location or globals().get("BQ_LOCATION") or "US"
    try:
        return pandas_gbq.read_gbq(sql, project_id=PROJECT_ID, dialect="standard", location=loc)
    except Exception as e:
        print("BigQuery error (location =", loc, "):\n", textwrap.fill(str(e), 120))
        if "was not found in location" in str(e):
            print("Hint: The dataset/table is in a different location than the job. "
                  "Ensure BQ_LOCATION matches the dataset’s location printed above.")
        raise


✅ Dataset found: mgmt590-465220:final_clean | location = US
Found 11 tables (first 25 shown): ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category_name_translation', 'products', 'products_raw', 'sellers', 'synthetic_fin_agent_sharing']


In [8]:
# Code Block -0 (always run first in a fresh Colab runtime)
!pip install -q jupyter-dash dash-bootstrap-components

import pandas as pd
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import html, dcc, Input, Output
import dash_bootstrap_components as dbc

print("✅ jupyter-dash and dependencies installed & imported")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.1 MB/s eta 0:00:00
✅ jupyter-dash and dependencies installed & imported


In [3]:
# Code Block S — Sanity checks (BigQuery & dataset wiring)

import sys, textwrap

# 1) Confirm globals from Code Block 0
try:
    print(f"PROJECT_ID={PROJECT_ID}, DATASET_ID={DATASET_ID}, BQ_LOCATION={BQ_LOCATION}")
except NameError as e:
    raise RuntimeError("Please run Code Block 0 first to define PROJECT_ID/DATASET_ID/BQ_LOCATION.") from e

# 2) Confirm table list (reuse TABLE_IDS if available, else list now)
try:
    table_ids = TABLE_IDS
except NameError:
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    ds = client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
    table_ids = [t.table_id for t in client.list_tables(ds)]

if not table_ids:
    raise RuntimeError("Dataset exists but no tables were found. Verify uploads to final_clean.")
print(f"Tables found (first 25): {table_ids[:25]}")

# 3) Simple query check
df_ok = run_gbq("SELECT 1 AS ok")
print("Simple query result:\n", df_ok)

# 4) Pick a test table (prefer orders, items, payments)
preferred = ["olist_orders_dataset", "olist_order_items_dataset", "olist_order_payments_dataset"]
test_table = next((t for t in preferred if t in table_ids), None) or table_ids[0]
print("Testing table:", test_table)

df_head = run_gbq(f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{test_table}` LIMIT 5")
print("Head from test table:\n", df_head)

# 5) Optional: small join test if both orders & items exist
needed = {"olist_orders_dataset", "olist_order_items_dataset"}
if needed.issubset(set(table_ids)):
    df_join = run_gbq(f"""
    SELECT
      o.order_id,
      COUNT(oi.order_item_id) AS items,
      MIN(CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS first_purchase_ts
    FROM `{PROJECT_ID}.{DATASET_ID}.olist_orders_dataset` o
    JOIN `{PROJECT_ID}.{DATASET_ID}.olist_order_items_dataset` oi
    USING(order_id)
    GROUP BY o.order_id
    ORDER BY items DESC
    LIMIT 5
    """)
    print("Join test sample:\n", df_join)
else:
    print("Join test skipped (orders/items not both present).")


PROJECT_ID=mgmt590-465220, DATASET_ID=final_clean, BQ_LOCATION=US
Tables found (first 25): ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category_name_translation', 'products', 'products_raw', 'sellers', 'synthetic_fin_agent_sharing']
Downloading: 100%|██████████|
Simple query result:
    ok
0   1
Testing table: customers
Downloading: 100%|██████████|
Head from test table:
                         customer_id                customer_unique_id  \
0  10ad09201fcc1c82d181ff7234bcdb3b  94742cd1fbac9146be7e2a139b63e13c   
1  5880e46677c68394bda62479fd673340  5dbba6c01268a8ad43f79157bf4454a0   
2  0f32385df13e46d88d997460208bc866  4f67110f6d6d1241111167b141bfa780   
3  dad907e170748a35ef4e92238b7308f3  36b1c0516f123351ffa87430416dcae5   
4  53996870173a3a001a1fb56ef00b9150  2624230437101e0bdcea1e48310d68a3   

   customer_zip_code_prefix customer_city customer_state  
0                     69900    rio branco             AC  
1             

In [4]:
# Code Block 0.5 — resolve table names (works with 'orders' or 'olist_orders_dataset') and set helpers

try:
    TABLE_IDS  # from Code Block 0
except NameError as e:
    raise RuntimeError("Run Code Block 0 first (it defines TABLE_IDS, PROJECT_ID, DATASET_ID, BQ_LOCATION, run_gbq).") from e

def resolve_table(*candidates: str) -> str:
    """Return the first candidate that exists in TABLE_IDS, else raise a helpful error."""
    for c in candidates:
        if c in TABLE_IDS:
            return c
    raise ValueError(f"None of the candidates {candidates} exist in dataset `{PROJECT_ID}.{DATASET_ID}`.\n"
                     f"Available tables: {TABLE_IDS}")

def fq(table_id: str) -> str:
    """Fully-qualified backticked table path."""
    return f"`{PROJECT_ID}.{DATASET_ID}.{table_id}`"

# Resolve common tables (prefers simplified IDs, falls back to olist_*)
ORDERS_ID       = resolve_table("orders", "olist_orders_dataset")
ORDER_ITEMS_ID  = resolve_table("order_items", "olist_order_items_dataset")
ORDER_PAYS_ID   = resolve_table("order_payments", "olist_order_payments_dataset")
ORDER_REVS_ID   = resolve_table("order_reviews", "olist_order_reviews_dataset")
CUSTOMERS_ID    = resolve_table("customers", "olist_customers_dataset")
PRODUCTS_ID     = resolve_table("products", "olist_products_dataset")
SELLERS_ID      = resolve_table("sellers", "olist_sellers_dataset")
GEO_ID          = resolve_table("geolocation", "olist_geolocation_dataset")
CAT_TR_ID       = resolve_table("product_category_name_translation", "product_category_name_translation")
# Optional / if present
SYNTH_ID        = "synthetic_fin_agent_sharing" if "synthetic_fin_agent_sharing" in TABLE_IDS else None

# Fully-qualified references for SQL
ORDERS      = fq(ORDERS_ID)
ORDER_ITEMS = fq(ORDER_ITEMS_ID)
ORDER_PAYS  = fq(ORDER_PAYS_ID)
ORDER_REVS  = fq(ORDER_REVS_ID)
CUSTOMERS   = fq(CUSTOMERS_ID)
PRODUCTS    = fq(PRODUCTS_ID)
SELLERS     = fq(SELLERS_ID)
GEO         = fq(GEO_ID)
CAT_TR      = fq(CAT_TR_ID)
SYNTH       = fq(SYNTH_ID) if SYNTH_ID else None

# Shared helpers
def cast_ts(expr: str) -> str:
    return f"CAST({expr} AS TIMESTAMP)"

REVENUE_EXPR = "(oi.price + oi.freight_value)"
DELIVERED_WHERE = "o.order_status = 'delivered'"


# Financial

In [9]:
# === Financial App (fixed filtering) ===
sql_fin_rev = f"""
SELECT
  DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
  c.customer_state,
  SUM({REVENUE_EXPR}) AS revenue
FROM {ORDERS} o
JOIN {ORDER_ITEMS} oi USING(order_id)
JOIN {CUSTOMERS} c USING(customer_id)
WHERE {DELIVERED_WHERE}
GROUP BY order_month, c.customer_state
ORDER BY order_month, c.customer_state
"""
df_fin_rev = run_gbq(sql_fin_rev)

sql_fin_aov = f"""
WITH per_order AS (
  SELECT
    o.order_id,
    DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
    c.customer_state,
    SUM({REVENUE_EXPR}) AS order_revenue
  FROM {ORDERS} o
  JOIN {ORDER_ITEMS} oi USING(order_id)
  JOIN {CUSTOMERS} c USING(customer_id)
  WHERE {DELIVERED_WHERE}
  GROUP BY o.order_id, order_month, c.customer_state
)
SELECT
  order_month, customer_state, AVG(order_revenue) AS aov, COUNT(*) AS orders
FROM per_order
GROUP BY order_month, customer_state
ORDER BY order_month, customer_state
"""
df_fin_aov = run_gbq(sql_fin_aov)

sql_fin_pay = f"""
SELECT
  DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
  c.customer_state,
  p.payment_type,
  SUM(p.payment_value) AS total_value
FROM {ORDER_PAYS} p
JOIN {ORDERS} o USING(order_id)
JOIN {CUSTOMERS} c USING(customer_id)
GROUP BY order_month, c.customer_state, p.payment_type
ORDER BY order_month, c.customer_state, total_value DESC
"""
df_fin_pay = run_gbq(sql_fin_pay)

sql_fin_status = f"""
SELECT
  DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
  c.customer_state,
  o.order_status,
  COUNT(*) AS orders
FROM {ORDERS} o
JOIN {CUSTOMERS} c USING(customer_id)
GROUP BY order_month, c.customer_state, o.order_status
ORDER BY order_month, c.customer_state
"""
df_fin_status = run_gbq(sql_fin_status)

import pandas as pd, plotly.express as px
from jupyter_dash import JupyterDash
from dash import html, dcc, Input, Output
import dash_bootstrap_components as dbc

# ✅ Force datetime for all frames used in filtering
for _df in (df_fin_rev, df_fin_aov, df_fin_pay, df_fin_status):
    if not _df.empty:
        _df["order_month"] = pd.to_datetime(_df["order_month"])

def months_from_frames(frames, col="order_month"):
    all_m = pd.concat([f[[col]] for f in frames if not f.empty], ignore_index=True).drop_duplicates()
    if all_m.empty:
        return [], {}, {}
    all_m = all_m.sort_values(col)
    months = list(pd.to_datetime(all_m[col]).unique())
    m2i = {m:i for i,m in enumerate(months)}
    i2m = {i:m for m,i in m2i.items()}
    return months, m2i, i2m

months, m2i, i2m = months_from_frames([df_fin_rev, df_fin_aov, df_fin_pay, df_fin_status])
if not months:
    raise RuntimeError("No data found for Financial app.")

states = sorted(
    pd.concat([
        df_fin_rev.get("customer_state", pd.Series([])),
        df_fin_aov.get("customer_state", pd.Series([])),
        df_fin_pay.get("customer_state", pd.Series([])),
        df_fin_status.get("customer_state", pd.Series([]))]).dropna().unique()
)

pay_types = sorted(df_fin_pay["payment_type"].dropna().unique()) if not df_fin_pay.empty else []
min_idx, max_idx = 0, len(months)-1

def _idx_range(idx_range):
    # robust to list/tuple/None
    if not idx_range:
        return min_idx, max_idx
    s, e = int(idx_range[0]), int(idx_range[1])
    return max(min_idx, s), min(max_idx, e)

def subset(df, sel_states, idx_range):
    if df.empty: return df
    s, e = _idx_range(idx_range)
    start, end = i2m[s], i2m[e]
    out = df.loc[df["order_month"].between(start, end)].copy()
    if sel_states:
        out = out[out["customer_state"].isin(sel_states)]
    return out

def subset_pay(df, sel_states, idx_range, sel_pay):
    out = subset(df, sel_states, idx_range)
    if sel_pay:
        out = out[out["payment_type"].isin(sel_pay)]
    return out

app_fin = JupyterDash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app_fin.layout = dbc.Container([
    html.H3("Financial Dashboard (Delivered Orders)"),
    dbc.Row([
        dbc.Col([
            html.Label("States"),
            dcc.Dropdown([{"label":s,"value":s} for s in states], id="fin-dd-states", value=[], multi=True, placeholder="All states")
        ], md=4),
        dbc.Col([
            html.Label("Payment types"),
            dcc.Dropdown([{"label":p,"value":p} for p in pay_types], id="fin-dd-pay", value=[], multi=True, placeholder="All types")
        ], md=4),
        dbc.Col([
            html.Label("Time range"),
            dcc.RangeSlider(id="fin-rs-months", min=min_idx, max=max_idx, value=[min_idx, max_idx],
                            marks={min_idx: months[min_idx].strftime("%Y-%m"),
                                   max_idx: months[max_idx].strftime("%Y-%m")})
        ], md=4)
    ], className="mt-2"),

    dbc.Row([
        dbc.Col(dcc.Graph(id="fin-revenue"), md=6),
        dbc.Col(dcc.Graph(id="fin-aov"), md=6),
    ], className="mt-2"),

    dbc.Row([
        dbc.Col(dcc.Graph(id="fin-paymix"), md=6),
        dbc.Col(dcc.Graph(id="fin-status"), md=6),
    ], className="mt-2 mb-4"),
], fluid=True)

@app_fin.callback(
    Output("fin-revenue","figure"),
    Output("fin-aov","figure"),
    Output("fin-paymix","figure"),
    Output("fin-status","figure"),
    Input("fin-dd-states","value"),
    Input("fin-dd-pay","value"),
    Input("fin-rs-months","value"),
)
def fin_update(sel_states, sel_pay, idx_range):
    dfr = subset(df_fin_rev, sel_states, idx_range)
    if dfr.empty:
        fig_rev = px.line(pd.DataFrame({"order_month":[],"revenue":[]}), x="order_month", y="revenue", title="Revenue — No data")
    else:
        rt = dfr.groupby("order_month", as_index=False)["revenue"].sum()
        fig_rev = px.line(rt, x="order_month", y="revenue", markers=True, title="Revenue Over Time")
        fig_rev.update_layout(yaxis_tickformat="$.2s")

    dfa = subset(df_fin_aov, sel_states, idx_range)
    if dfa.empty:
        fig_aov = px.line(pd.DataFrame({"order_month":[],"aov":[]}), x="order_month", y="aov", title="AOV — No data")
    else:
        at = dfa.groupby("order_month", as_index=False)["aov"].mean()
        fig_aov = px.line(at, x="order_month", y="aov", markers=True, title="AOV Over Time")
        fig_aov.update_layout(yaxis_tickformat="$.2f")

    dfp = subset_pay(df_fin_pay, sel_states, idx_range, sel_pay)
    if dfp.empty:
        fig_pay = px.bar(pd.DataFrame({"payment_type":[],"total_value":[]}), x="payment_type", y="total_value", title="Payment Mix — No data")
    else:
        pm = dfp.groupby("payment_type", as_index=False)["total_value"].sum().sort_values("total_value", ascending=False)
        fig_pay = px.bar(pm, x="payment_type", y="total_value", title="Payment Mix (Value)")
        fig_pay.update_layout(yaxis_tickformat="$.2s")

    dfs = subset(df_fin_status, sel_states, idx_range)
    if dfs.empty:
        fig_st = px.bar(pd.DataFrame({"order_status":[],"orders":[]}), x="order_status", y="orders", title="Order Status — No data")
    else:
        st = dfs.groupby("order_status", as_index=False)["orders"].sum().sort_values("orders", ascending=False)
        fig_st = px.bar(st, x="order_status", y="orders", title="Order Status Distribution")

    return fig_rev, fig_aov, fig_pay, fig_st

_run_kwargs = dict(port=8051, debug=False)
try:
    app_fin.run_server(mode="inline", **_run_kwargs)
except AttributeError:
    app_fin.run(jupyter_mode="inline", **_run_kwargs)


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


<IPython.core.display.Javascript object>

# Customer/Market


In [10]:
# === Customer/Market App (confirmed filtering) ===
sql_cm = f"""
WITH revenue AS (
  SELECT
    DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
    c.customer_state,
    COALESCE(e.product_category_name_english, p.product_category_name) AS category_en,
    SUM({REVENUE_EXPR}) AS revenue
  FROM {ORDERS} o
  JOIN {ORDER_ITEMS} oi USING(order_id)
  JOIN {CUSTOMERS} c USING(customer_id)
  JOIN {PRODUCTS} p ON p.product_id = oi.product_id
  LEFT JOIN {CAT_TR} e USING(product_category_name)
  WHERE {DELIVERED_WHERE}
  GROUP BY order_month, c.customer_state, category_en
)
SELECT * FROM revenue
"""
df_cm = run_gbq(sql_cm)

import pandas as pd, plotly.express as px
from jupyter_dash import JupyterDash
from dash import html, dcc, Input, Output
import dash_bootstrap_components as dbc

if df_cm.empty:
    raise RuntimeError("No delivered revenue for Customer/Market app.")

# ✅ force datetime
df_cm["order_month"] = pd.to_datetime(df_cm["order_month"])
df_cm["category_en"] = df_cm["category_en"].fillna("Unknown")

months = sorted(df_cm["order_month"].unique())
m2i = {m:i for i,m in enumerate(months)}
i2m = {i:m for m,i in m2i.items()}
min_idx, max_idx = 0, len(months)-1
states = sorted(df_cm["customer_state"].dropna().unique())
cats = sorted(df_cm["category_en"].dropna().unique())

def _idx_range(idx_range):
    if not idx_range: return min_idx, max_idx
    s, e = int(idx_range[0]), int(idx_range[1])
    return max(min_idx, s), min(max_idx, e)

def subset(df, sel_states, sel_cats, idx_range):
    s, e = _idx_range(idx_range)
    m_start, m_end = i2m[s], i2m[e]
    out = df.loc[df["order_month"].between(m_start, m_end)].copy()
    if sel_states:
        out = out[out["customer_state"].isin(sel_states)]
    if sel_cats:
        out = out[out["category_en"].isin(sel_cats)]
    return out

app_cm = JupyterDash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app_cm.layout = dbc.Container([
    html.H3("Customer / Market Dashboard"),
    dbc.Row([
        dbc.Col([
            html.Label("States"),
            dcc.Dropdown([{"label":s,"value":s} for s in states],
                         id="cm-dd-states", value=[], multi=True, placeholder="All states")
        ], md=4),
        dbc.Col([
            html.Label("Categories"),
            dcc.Dropdown([{"label":c,"value":c} for c in cats],
                         id="cm-dd-cats", value=[], multi=True, placeholder="All categories")
        ], md=5),
        dbc.Col([
            html.Label("Time range"),
            dcc.RangeSlider(id="cm-rs-months",
                            min=min_idx, max=max_idx, value=[min_idx, max_idx],
                            marks={min_idx: months[min_idx].strftime("%Y-%m"),
                                   max_idx: months[max_idx].strftime("%Y-%m")})
        ], md=3),
    ], className="mt-2"),
    dbc.Row([
        dbc.Col(dcc.Graph(id="cm-state"), md=6),
        dbc.Col(dcc.Graph(id="cm-cat"), md=6),
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id="cm-trend"), md=12),
    ], className="mt-2 mb-4"),
], fluid=True)

@app_cm.callback(
    Output("cm-state","figure"),
    Output("cm-cat","figure"),
    Output("cm-trend","figure"),
    Input("cm-dd-states","value"),
    Input("cm-dd-cats","value"),
    Input("cm-rs-months","value"),
)
def cm_update(sel_states, sel_cats, idx_range):
    dff = subset(df_cm, sel_states or [], sel_cats or [], idx_range)

    if dff.empty:
        fig_s = px.bar(pd.DataFrame({"customer_state":[], "revenue":[]}), x="customer_state", y="revenue",
                       title="Revenue by State — No data")
        fig_c = px.bar(pd.DataFrame({"category_en":[], "revenue":[]}), x="category_en", y="revenue",
                       title="Revenue by Category — No data")
        fig_t = px.line(pd.DataFrame({"order_month":[], "revenue":[]}), x="order_month", y="revenue",
                        title="Revenue Over Time — No data")
        return fig_s, fig_c, fig_t

    sg = dff.groupby("customer_state", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False).head(25)
    fig_s = px.bar(sg, x="customer_state", y="revenue", title="Revenue by State")
    fig_s.update_layout(yaxis_tickformat="$.2s")

    cg = dff.groupby("category_en", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False).head(25)
    fig_c = px.bar(cg, x="category_en", y="revenue", title="Revenue by Category")
    fig_c.update_layout(xaxis_tickangle=-30, yaxis_tickformat="$.2s")

    tg = dff.groupby("order_month", as_index=False)["revenue"].sum()
    fig_t = px.line(tg, x="order_month", y="revenue", markers=True, title="Revenue Over Time")
    fig_t.update_layout(yaxis_tickformat="$.2s")

    return fig_s, fig_c, fig_t

_run_kwargs = dict(port=8052, debug=False)
try:
    app_cm.run_server(mode="inline", **_run_kwargs)
except AttributeError:
    app_cm.run(jupyter_mode="inline", **_run_kwargs)


Downloading: 100%|██████████|


<IPython.core.display.Javascript object>

# Operations


In [12]:
# --- Queries (per-order base, category map, reviews) ---
sql_ops_base = f"""
SELECT
  o.order_id,
  DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
  c.customer_state,
  TIMESTAMP_DIFF(
    {cast_ts('o.order_delivered_customer_date')},
    {cast_ts('o.order_estimated_delivery_date')},
    DAY
  ) AS delay_days,
  CASE
    WHEN {cast_ts('o.order_delivered_customer_date')} IS NULL
         OR {cast_ts('o.order_estimated_delivery_date')} IS NULL THEN NULL
    WHEN {cast_ts('o.order_delivered_customer_date')} <= {cast_ts('o.order_estimated_delivery_date')}
      THEN 1 ELSE 0 END AS on_time
FROM {ORDERS} o
JOIN {CUSTOMERS} c USING(customer_id)
WHERE o.order_status = 'delivered'
  AND {cast_ts('o.order_delivered_customer_date')} IS NOT NULL
  AND {cast_ts('o.order_estimated_delivery_date')} IS NOT NULL
"""
df_ops_base = run_gbq(sql_ops_base)

sql_ops_map = f"""
SELECT DISTINCT
  oi.order_id,
  COALESCE(e.product_category_name_english, p.product_category_name) AS category_en
FROM {ORDER_ITEMS} oi
JOIN {PRODUCTS} p ON p.product_id = oi.product_id
LEFT JOIN {CAT_TR} e USING(product_category_name)
"""
df_map = run_gbq(sql_ops_map)

sql_ops_reviews = f"""
SELECT order_id, review_score, {cast_ts('review_creation_date')} AS review_ts
FROM {ORDER_REVS}
WHERE review_score IS NOT NULL
"""
df_reviews = run_gbq(sql_ops_reviews)

# --- Guards & preprocessing ---
if df_ops_base.empty:
    raise RuntimeError("No delivered orders with delivery timestamps available for Operations dashboard.")

df_ops_base["order_month"] = pd.to_datetime(df_ops_base["order_month"])
df_map["category_en"] = df_map["category_en"].fillna("Unknown")

order_to_cats = df_map.groupby("order_id")["category_en"].apply(set).to_dict()
all_states = sorted(df_ops_base["customer_state"].dropna().unique())
all_cats = sorted(df_map["category_en"].dropna().unique())

months = sorted(df_ops_base["order_month"].unique())
m2i = {m:i for i,m in enumerate(months)}
i2m = {i:m for m,i in m2i.items()}
min_idx, max_idx = 0, len(months)-1

# --- Helpers ---
CHART_H = 420

def _fix_fig(fig, h=CHART_H):
    return fig.update_layout(height=h, autosize=False,
                             margin=dict(t=48, b=40, l=40, r=20))

def _idx_range(idx_range, min_i=min_idx, max_i=max_idx):
    if not idx_range: return min_i, max_i
    s, e = int(idx_range[0]), int(idx_range[1])
    return max(min_i, s), min(max_i, e)

def filter_orders(sel_states, sel_cats, idx_range):
    s, e = _idx_range(idx_range)
    start, end = i2m[s], i2m[e]
    df = df_ops_base.loc[df_ops_base["order_month"].between(start, end)].copy()
    if sel_states:
        df = df[df["customer_state"].isin(sel_states)]
    if sel_cats:
        sel_c = set(sel_cats)
        mask = df["order_id"].map(lambda oid: bool(order_to_cats.get(oid, set()) & sel_c))
        df = df[mask]
    return df

# --- App layout ---
app_ops = JupyterDash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app_ops.layout = dbc.Container([
    html.H3("Operations Dashboard"),
    dbc.Row([
        dbc.Col([
            html.Label("States"),
            dcc.Dropdown([{"label":s,"value":s} for s in all_states],
                         id="ops-dd-states", value=[], multi=True, placeholder="All states")
        ], md=4),
        dbc.Col([
            html.Label("Categories (orders that contain any selection)"),
            dcc.Dropdown([{"label":c,"value":c} for c in all_cats],
                         id="ops-dd-cats", value=[], multi=True, placeholder="All categories")
        ], md=5),
        dbc.Col([
            html.Label("Time range"),
            dcc.RangeSlider(id="ops-rs-months", min=min_idx, max=max_idx, value=[min_idx, max_idx],
                            marks={min_idx: months[min_idx].strftime("%Y-%m"),
                                   max_idx: months[max_idx].strftime("%Y-%m")})
        ], md=3),
    ], className="mt-2"),

    dbc.Row([
        dbc.Col(dcc.Graph(id="ops-ontime",
                          style={'height': f'{CHART_H}px'},
                          config={'responsive': False}), md=6),
        dbc.Col([
            html.Div("Delay histogram bins"),
            dcc.Slider(id="ops-bins", min=10, max=120, step=5, value=60,
                       marks={10:"10", 60:"60", 120:"120"}),
            dcc.Graph(id="ops-delayhist",
                      style={'height': f'{CHART_H}px'},
                      config={'responsive': False})
        ], md=6),
    ], className="mt-2"),

    dbc.Row([
        dbc.Col(dcc.Graph(id="ops-reviewdist",
                          style={'height': f'{CHART_H}px'},
                          config={'responsive': False}), md=6),
        dbc.Col(dcc.Graph(id="ops-delay-vs-review",
                          style={'height': f'{CHART_H}px'},
                          config={'responsive': False}), md=6),
    ], className="mt-2 mb-4"),
], fluid=True, style={'maxWidth': '1200px'})

# --- Callbacks ---
@app_ops.callback(
    Output("ops-ontime","figure"),
    Output("ops-delayhist","figure"),
    Output("ops-reviewdist","figure"),
    Output("ops-delay-vs-review","figure"),
    Input("ops-dd-states","value"),
    Input("ops-dd-cats","value"),
    Input("ops-rs-months","value"),
    Input("ops-bins","value"),
)
def ops_update(sel_states, sel_cats, idx_range, bins):
    dff = filter_orders(sel_states or [], sel_cats or [], idx_range)

    # On-time rate over time
    if dff.empty:
        fig_ot = px.line(pd.DataFrame({"order_month":[],"on_time_rate":[]}),
                         x="order_month", y="on_time_rate", title="On-time Rate — No data")
    else:
        ot = dff.groupby("order_month", as_index=False)\
                .agg(on_time_rate=("on_time", lambda s: np.nanmean(s==1)))
        fig_ot = px.line(ot, x="order_month", y="on_time_rate", markers=True, title="On-time Rate Over Time")
        fig_ot.update_layout(yaxis_tickformat=".0%")
    _fix_fig(fig_ot)

    # Delay histogram
    if dff.empty or dff["delay_days"].dropna().empty:
        fig_hist = px.histogram(pd.DataFrame({"delay_days":[]}), x="delay_days", nbins=int(bins),
                                title="Delivery Delay Distribution — No data")
    else:
        fig_hist = px.histogram(dff.dropna(subset=["delay_days"]), x="delay_days", nbins=int(bins),
                                title="Delivery Delay Distribution (Days)")
        fig_hist.add_vline(x=0, line_dash="dot")
    _fix_fig(fig_hist)

    # Reviews
    if dff.empty or df_reviews.empty:
        fig_rdist = px.bar(pd.DataFrame({"review_score":[],"n":[]}), x="review_score", y="n",
                           title="Review Score Distribution — No data")
        fig_box = px.box(pd.DataFrame({"review_score":[],"delay_days":[]}),
                         x="review_score", y="delay_days",
                         title="Delay vs Review — No data")
    else:
        sel_orders = set(dff["order_id"].unique())
        dr = df_reviews[df_reviews["order_id"].isin(sel_orders)].copy()
        if dr.empty:
            fig_rdist = px.bar(pd.DataFrame({"review_score":[],"n":[]}), x="review_score", y="n",
                               title="Review Score Distribution — No data")
            fig_box = px.box(pd.DataFrame({"review_score":[],"delay_days":[]}),
                             x="review_score", y="delay_days",
                             title="Delay vs Review — No data")
        else:
            rdist = dr.groupby("review_score").size().reset_index(name="n")
            fig_rdist = px.bar(rdist, x="review_score", y="n", title="Review Score Distribution")

            merged = dr.merge(dff[["order_id","delay_days"]], on="order_id", how="left").dropna(subset=["delay_days"])
            if merged.empty:
                fig_box = px.box(pd.DataFrame({"review_score":[],"delay_days":[]}),
                                 x="review_score", y="delay_days",
                                 title="Delay vs Review — No data")
            else:
                fig_box = px.box(merged, x="review_score", y="delay_days",
                                 title="Delivery Delay vs Review Score")
                fig_box.add_vline(x=2.5, line_dash="dot")
    _fix_fig(fig_rdist)
    _fix_fig(fig_box)

    return fig_ot, fig_hist, fig_rdist, fig_box

# --- Run (version-safe) ---
_run_kwargs = dict(port=8053, debug=False)
try:
    app_ops.run_server(mode="inline", **_run_kwargs)
except AttributeError:
    app_ops.run(jupyter_mode="inline", **_run_kwargs)

Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|


<IPython.core.display.Javascript object>

# More Charts


In [17]:
# Run after Code Block 0 and 0.5 (uses run_gbq, ORDERS, ORDER_ITEMS, CUSTOMERS, PRODUCTS, CAT_TR, cast_ts, REVENUE_EXPR, DELIVERED_WHERE)

import plotly.express as px
import pandas as pd

sql_treemap = f"""
WITH revenue AS (
  SELECT
    DATE_TRUNC(DATE({cast_ts('o.order_purchase_timestamp')}), MONTH) AS order_month,
    c.customer_state,
    COALESCE(e.product_category_name_english, p.product_category_name) AS category_en,
    SUM({REVENUE_EXPR}) AS revenue
  FROM {ORDERS} o
  JOIN {ORDER_ITEMS} oi USING(order_id)
  JOIN {CUSTOMERS} c USING(customer_id)
  JOIN {PRODUCTS} p ON p.product_id = oi.product_id
  LEFT JOIN {CAT_TR} e USING(product_category_name)
  WHERE {DELIVERED_WHERE}
  GROUP BY order_month, c.customer_state, category_en
)
SELECT * FROM revenue
"""
df_tree = run_gbq(sql_treemap)
if df_tree.empty:
    print("No data for treemap.")
else:
    df_tree["category_en"] = df_tree["category_en"].fillna("Unknown")
    fig = px.treemap(
        df_tree.groupby(["customer_state","category_en"], as_index=False)["revenue"].sum(),
        path=["customer_state","category_en"],
        values="revenue",
        title="Drill-in Treemap: Revenue by State → Category (Delivered Orders)"
    )
    fig.update_traces(root_color="lightgray")
    fig.show()


Downloading: 100%|██████████|


In [18]:
!pip -q install jupyter-dash dash dash-bootstrap-components


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.2 MB/s eta 0:00:00
